## Importing required libraries


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")


In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
# df_silver = df_silver.select("customer_id","city","customer_name","email","signup_date","state")
# df_silver.show()

In [0]:
# ================================= QUALITY CHECK ======================

# 01 checking city columm
df_bronze.select("city").distinct()

# Checking null count in each column 
null_counts = df_bronze.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_bronze.columns])

## checking email column
df_bronze.filter(~F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))

### checking signup_date coluumn
df_bronze.count()
df_bronze.filter(F.length(F.col("signup_date")) > 10)

### checking state column
df_bronze.select("state").distinct().count()

### check duplicate in customer_id
df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)

# Note : i skipped the action command becouse i run it once before if you are first time running this code then add action comman for it 

In [0]:

corrections = {"Bengalore": "Bengaluru", "Bombay": "Mumbai" }

# 2. Apply the transformation
df_silver = df_bronze.withColumn("city", 
    F.trim(
        F.initcap(
            F.coalesce(F.create_map([F.lit(x) for x in sum(corrections.items(), ())])[F.col("city")], F.col("city"))
        )
    )
).fillna("Unknown", subset=["city"]) # Step D: Handle NULLs

df_silver.show()

In [0]:
df_silver = df_silver.withColumn("customer_name",
        F.trim("customer_name")                                 
)
df_silver.show()




In [0]:
# valid_email_condition = 
valid_email_condition = F.col("email").rlike(
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)

df_valid = df_silver.filter(valid_email_condition)
df_invalid = df_silver.filter(~valid_email_condition)
df_silver = df_valid

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse_project.quarantine;

In [0]:
df_invalid.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.quarantine.bad_customers_email"
    )

In [0]:
valid_signup_date_condition =  F.col("signup_date")  < F.current_date()
df_valid_signup_date = df_silver.filter(valid_signup_date_condition)

df_invalid_date = df_silver.filter(~valid_signup_date_condition)
df_silver = df_valid_signup_date

df_invalid_date.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.quarantine.bad_customers_signup_date"
    )

In [0]:
df_silver = df_silver.dropDuplicates(["customer_id"])
if not (spark.catalog.tableExists(f"{catalog}.{silver_schema}.{data_source}")):
    df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalog}.{silver_schema}.{data_source}"
    )
    print("sucessfully writed  data to delta location")
else:
    print("Doing upsert operation")
    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{silver_schema}.{data_source}"
    )

    delta_table.alias("target").merge(
        source=df_silver.alias("source"),
        condition="""
            target.customer_id = source.customer_id
        """
    ).whenMatchedUpdate(
        set={
             "customer_name": "coalesce(source.customer_name, target.customer_name)",
            "city": "coalesce(source.city, target.city)",
            "state": "coalesce(source.state, target.state)",
            "email": "coalesce(source.email, target.email)",
            "signup_date": "coalesce(source.signup_date, target.signup_date)"
            
        }
    ).whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "city": "source.city",
            "state": "source.state",
            "email": "source.email",
            "signup_date":"source.signup_date"
        }
    ).execute()
